In [2]:
# ============================================================
# TP : Programmation parallèle avec Numba
# Objectif : Calculer la moyenne pondérée des notes d'étudiants
# ============================================================

# --- Importation des bibliothèques ---
# pandas : manipulation de données tabulaires
# numpy  : calcul numérique efficace
# time   : mesure des temps d'exécution
# os     : récupération du nombre de cœurs disponibles
# numba  : compilation JIT et parallélisation
import pandas as pd
import numpy as np
import time
import os
from numba import njit, prange

# --- Chargement des données ---
# Le fichier CSV "notes_etudiants.csv" contient 1 000 000 étudiants
# avec leurs identifiants, prénoms, noms et notes en Maths, Physique, Anglais.
df = pd.read_csv("notes_etudiants.csv")

# Conversion des colonnes en tableaux NumPy pour un calcul plus rapide
maths = df["Maths"].values
physique = df["Physique"].values
anglais = df["Anglais"].values

# ============================================================
# 1. Version séquentielle
# ============================================================
# Définition d'une fonction simple qui calcule la moyenne pondérée
# Coefficients : Maths=5, Physique=4, Anglais=2
def moyenne_seq(m, p, a):
    return (m*5 + p*4 + a*2) / 11

# Mesure du temps d'exécution séquentiel
start = time.time()
moy_seq = [moyenne_seq(m, p, a) for m, p, a in zip(maths, physique, anglais)]
t_seq = time.time() - start
print("Temps séquentiel :", t_seq, "secondes")

# ============================================================
# 2. Version parallèle avec Numba
# ============================================================
# Utilisation de @njit(parallel=True) pour paralléliser la boucle
# prange permet de distribuer les itérations sur plusieurs cœurs
@njit(parallel=True)
def moyenne_parallel(maths, physique, anglais):
    N = len(maths)
    moyennes = np.empty(N)
    for i in prange(N):
        moyennes[i] = (maths[i]*5 + physique[i]*4 + anglais[i]*2) / 11
    return moyennes

# Premier appel : compilation JIT (non mesuré)
_ = moyenne_parallel(maths, physique, anglais)

# Mesure du temps parallèle (après compilation)
start = time.time()
moy_parallel = moyenne_parallel(maths, physique, anglais)
t_par = time.time() - start
print("Temps parallèle :", t_par, "secondes")

# ============================================================
# 3. Calcul du Speedup
# ============================================================
# Speedup = Temps séquentiel / Temps parallèle
speedup = t_seq / t_par
print("Speedup =", speedup)

# ============================================================
# 4. Loi d'Amdahl : proportion parallélisable
# ============================================================
# Formule : P = (1 - 1/S) / (1 - 1/N)
# S = speedup mesuré
# N = nombre de cœurs disponibles
N = os.cpu_count()
P = (1 - 1/speedup) / (1 - 1/N)

print("Nombre de cœurs utilisés :", N)
print("Proportion parallélisable (P) =", P*100, "%")

# ============================================================
# Commentaires académiques :
# - La version séquentielle illustre le calcul classique en Python.
# - La version parallèle exploite Numba pour distribuer les calculs
#   sur plusieurs cœurs, ce qui réduit fortement le temps d'exécution.
# - Le speedup mesuré peut dépasser largement le nombre de cœurs
#   car Numba optimise aussi le code (compilation JIT).
# - La loi d'Amdahl permet d'estimer la proportion du programme
#   réellement parallélisable. Une valeur >100% indique que
#   l'accélération provient à la fois de la parallélisation
#   et de l'optimisation du code compilé.
# ============================================================


Temps séquentiel : 2.0134150981903076 secondes
Temps parallèle : 0.0052106380462646484 secondes
Speedup = 386.40471288034774
Nombre de cœurs utilisés : 8
Proportion parallélisable (P) = 113.98994741104602 %
